In [ ]:
import sys,os,glob
sys.path.append(r'D:\python\neuron-vis\neuronVis')
import numpy as np
import pandas as pd
import BoundLaplace
import json
from PIL import Image
# import IONData as IONData
import nrrd
import Flatmap

In [ ]:
from Flatmap import map2Flatmap
import joblib

In [ ]:
flatenPara = joblib.load(r'D:\python\neuron-vis\resource\flatenPara.pkl')

In [ ]:
import os
import nrrd

# 定义本地资源目录（使用原始字符串 r"..." 避免 Windows 路径中的反斜杠转义问题）
resource_dir = r"D:\python\neuron-vis\resource"

# 直接从本地路径读取各 NRRD 文件
grid, header = nrrd.read(os.path.join(resource_dir, "boundlaplace20.nrrd"))
relaxation, relaxationheader = nrrd.read(os.path.join(resource_dir, "boundlaplaceout20.nrrd"))

dv0, dv0header = nrrd.read(os.path.join(resource_dir, "dv0.nrrd"))
dv1, dv1header = nrrd.read(os.path.join(resource_dir, "dv1.nrrd"))
dv2, dv2header = nrrd.read(os.path.join(resource_dir, "dv2.nrrd"))

In [ ]:
os.chdir(r"J:\BLA_four_types\csv_PFC")

In [ ]:
files = os.listdir()

In [ ]:
files

In [ ]:
## get point
os.makedirs("..\\csv_PFC_flat",exist_ok=True)
for file in files:

    df = pd.read_csv(file,index_col=0)
    # df1 = df.loc[df.name_use.isin(["PL", "ORB", "ACA", "ILA", "FRP","MO","AI"])&df.terminal == 1]
    df1 = df.loc[df.name_use.isin(["ORB"])&df.terminal == 1]
    df2= df1[["AP","DV","ML"]]
    soma_xyz = np.array(df2,dtype=float)
    soma_xyz_new = soma_xyz/20
    # soma_xyz_10 = soma_xyz/10
    point=[]
    for p in soma_xyz_new:
        
        if p[2]>285:
            z=285*2-p[2]
        else:
            z=p[2]
        point.append([p[0],p[1],z])

    dv0=dv0.astype(np.float32)/1000-1
    dv1=dv1.astype(np.float32)/1000-1
    dv2=dv2.astype(np.float32)/1000-1
    pointjson = {}
    out=BoundLaplace.ComputeStreamlines(grid,dv0,dv1,dv2,point)
    index=0
    for p in out:
        p2d = map2Flatmap(flatenPara,np.array(p[1])*2,True)
        # print("point:",index,p[0],p2d)
        pointjson[index]=[p2d[0],p2d[1]]
        index+=1

    dftmp = pd.DataFrame(pointjson).T
    dftmp.columns = ["flat_x","flat_y"]
    dfall = pd.concat([df1.reset_index(drop=True), 
                        dftmp.reset_index(drop=True)], 
                    axis=1)
    dfall.to_csv("..\\csv_PFC_flat\\"+file)



In [22]:
dfall

,ID,type,AP,DV,ML,R,parent,area_ID,area_name,name_use,bratch_ID,side,terminal,flat_x,flat_y
0,19250,2,3134.08,3910.30,5414.46,1.000000,19249,620,ORBm5,ORB,131,left,1,869.179912,324.000904
1,19318,2,3064.70,4040.76,5350.98,1.000000,19317,620,ORBm5,ORB,132,left,1,869.868932,311.976095
2,20902,2,2817.90,3760.94,5641.66,2.250000,20901,484,ORBm1,ORB,149,left,1,926.213723,309.993780
3,20926,2,3155.18,4060.70,5510.58,1.000000,20925,582,ORBm2/3,ORB,151,left,1,876.631529,315.016898
4,20985,2,3129.32,4061.76,5391.66,0.750000,20984,620,ORBm5,ORB,152,left,1,865.935244,316.474878
5,20993,2,2752.04,3353.30,5620.90,1.000000,20992,484,ORBm1,ORB,153,left,1,921.030806,340.678170
6,21513,2,3104.84,3957.98,5575.40,0.750000,21512,582,ORBm2/3,ORB,160,left,1,884.928864,320.462631
7,22147,2,2846.34,3420.24,5544.74,1.417969,22146,582,ORBm2/3,ORB,164,left,1,906.497106,340.171373
8,22495,2,2755.12,3338.04,5646.88,1.000000,22494,484,ORBm1,ORB,169,left,1,924.454533,343.666713
9,22603,2,2839.16,3552.60,5431.78,0.917969,22602,620,ORBm5,ORB,172,left,1,894.330796,329.360328


In [23]:
import matplotlib.pyplot as plt
import copy

In [24]:
files = glob.glob("..\\csv_PFC_flat\\*")

In [31]:
# Hex 格式字符串
colors_hex = [
    '#B43C28',  # 砖红
    '#D27D1E',  # 暖橙
    '#C8C332',  # 芥末黄
    '#3C9B46',  # 翠绿
    '#32AAB9',  # 青蓝
    '#4B78C8',  # 钢蓝
    '#9150C3',  # 柔紫
    '#C850A0'   # 玫瑰粉
]

In [32]:
dfall = pd.DataFrame()
for file,c in zip(files,colors_hex):
# for file in files:    
    dfc = pd.read_csv(file,index_col=0)
    dft = dfc[["flat_x","flat_y","name_use"]]
    dft["file"] = file.split("\\")[-1][0:-4]
    dft["color"] = c
    dfall = pd.concat([dfall,dft],axis=0)

C:\Users\zljia\AppData\Local\Temp\ipykernel_19040\3323162302.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dft["file"] = file.split("\\")[-1][0:-4]
C:\Users\zljia\AppData\Local\Temp\ipykernel_19040\3323162302.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dft["color"] = c


In [26]:
dfall.name_use.unique()

array(['ORB'], dtype=object)

In [ ]:
files

In [33]:
def original(i,j,ksize,img):
    #找到矩阵坐标
    x1=y1=-ksize//2
    x2=y2=ksize+x1
    temp=np.zeros(ksize*ksize)
    count=0
    #处理图像
    for m in range(x1,x2):
        for n in range(y1,y2):
            if i+m<0 or i+m>img.shape[0]-1 or j+n<0 or j+n>img.shape[1]-1:
                temp[count]=img[i,j]
            else:
                temp[count]=img[i+m,j+n]
            count +=1
    return temp

#自定义最大值最小值滤波器
def max_min_function(ksize,img,flag):
    img0=copy.copy(img)
    for i in range(0,img.shape[0]):
        for j in range(2,img.shape[1]):
        
            temp=original(i,j,ksize,img0)
            if flag ==0: #设置flag参数，如果0就检测最大值，如果是1检测最小值
                img[i,j]=np.max(temp)
            elif flag==1:
                img[i,j]=np.min(temp)
    return img

In [28]:
dfall.name_use.unique()

array(['ORB'], dtype=object)

In [34]:
import cv2
imag_flat,option = nrrd.read(r"D:\python\neuron-vis\resource\flatmap.nrrd")

imag_flat[imag_flat == 484] = 4840
imag_flat[imag_flat == 448] = 4480
blur=max_min_function(3,imag_flat.T,0)
blur = blur.astype(np.uint8)
canny = cv2.Canny(blur,10,20)
bitwise = cv2.bitwise_not(canny)
# cv2.imwrite(r'J:\BLA_four_types\flat\SSp_soma_flat-7.png',bitwise)
fig = plt.figure(figsize=(16,16),dpi=600)
ax = fig.add_subplot(1,1,1)
ax.imshow(bitwise,cmap = 'gray')
for file_name, group in dfall.groupby('file', sort=False):
    # 取出当前分组对应的颜色
    point_color = group['color'].iloc[0]

    ax.scatter(
        group['flat_x'],
        group['flat_y'],
        c=point_color,
        label=file_name,
        s=15,          # 点的大小，可按需调节 (如 10~30)
        alpha=1,     # 点的透明度
        edgecolors='none'
    )

# 设置图像属性与反转 Y 轴（如果 flatmap 坐标系原点在左上方，取消下一行注释）
# ax.invert_yaxis()

ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('Soma Distribution Flatmap', fontsize=14)
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 坐标比例

# 添加图例（放在图右侧外部，防止遮挡散点）
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=9,
    markerscale=1.5
)

# 去除多余的顶部和右侧边框 (美化样式)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

plt.savefig(r'J:\BLA_four_types\flat\terminal_flat-6.png',dpi=300)

C:\Users\zljia\AppData\Local\Temp\ipykernel_19040\2574932154.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [ ]:
dfall.name_use.unique()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'name_use', 'file', 'color']

# 1. 过滤无效坐标 (0, 0)
df_valid = dfall[(dfall['flat_x'] > 0) & (dfall['flat_y'] > 0)].copy()

# 2. 定义 5 个脑区专属的高区分度柔和颜色字典 (RGB888 Hex)
target_regions = ['ORB', 'PL', 'FRP', 'ACA', 'ILA',"AI","MO"]
region_colors = {
    'ORB': '#B43C28',  # 砖红
    'PL':  '#3C9B46',  # 翠绿
    'FRP': '#4B78C8',  # 钢蓝
    'ACA': '#D27D1E',  # 暖橙
    'ILA': '#9150C3',   # 柔紫
    'AI': "#1E93D2",  # 暖橙
    'MO': "#63C350"   # 柔紫
}

# 3. 筛选目标脑区数据
df_target = df_valid[df_valid['name_use'].isin(target_regions)]

# 4. 创建画板并绘制
fig, ax = plt.subplots(figsize=(16, 16), dpi=300)

for region in target_regions:
    sub_df = df_target[df_target['name_use'] == region]
    if sub_df.empty:
        continue
    ax.imshow(bitwise,cmap = 'gray')
    ax.scatter(
        sub_df['flat_x'],
        sub_df['flat_y'],
        c=region_colors[region],
        label=region,
        s=18,           # 点大小
        alpha=0.75,     # 透明度（略微透明方便观察区域重叠与过渡）
        edgecolors='none'
    )

# 5. 坐标轴与样式配置
ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('PFC Subregions Distribution on Flatmap', fontsize=14, fontweight='bold')
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 空间比例

# 如果图像坐标系原点在左上（像素坐标），取消下面这行注释：
# ax.invert_yaxis()

# 6. 图例设置（按脑区标注）
ax.legend(
    title='Brain Region',
    bbox_to_anchor=(1.03, 1),
    loc='upper left',
    frameon=False,
    fontsize=10,
    title_fontsize=11,
    markerscale=1.6
)

# 隐藏上方和右侧边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
save_path = os.path.join(output_dir, f'flatmap.png')
plt.savefig(save_path, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {save_path}')

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'name_use', 'file', 'color']

output_dir = r'J:\BLA_four_types\flat\regions_split'
os.makedirs(output_dir, exist_ok=True)

# 1. 过滤掉无效原点坐标 (0, 0)，避免拉偏视图范围
df_valid = dfall[(dfall['flat_x'] > 0) & (dfall['flat_y'] > 0)].copy()

# 2. 按照 name_use 脑区分组出图
for region_name, df_region in df_valid.groupby('name_use', sort=True):
    
    fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
    ax.imshow(bitwise,cmap = 'gray')
    
    # 在当前脑区内，按 file 分组绘制散点并应用对应的 color
    for file_name, group in df_region.groupby('file', sort=False):
        point_color = group['color'].iloc[0]
        
        ax.scatter(
            group['flat_x'],
            group['flat_y'],
            c=point_color,
            label=file_name,
            s=20,
            alpha=0.85,
            edgecolors='none'
        )
    
    # 图像属性配置
    ax.set_title(f'Flatmap - {region_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Flat X', fontsize=11)
    ax.set_ylabel('Flat Y', fontsize=11)
    ax.set_aspect('equal', adjustable='box')  # 保持 1:1 物理比例
    
    # 如果 flatmap 的 Y 轴是从上往下递增的像素坐标，取消下面这行注释：
    # ax.invert_yaxis()
    
    # 图例配置（放在图外侧右边）
    ax.legend(
        bbox_to_anchor=(1.05, 1),
        loc='upper left',
        frameon=False,
        fontsize=8,
        markerscale=1.4
    )
    
    # 隐藏上方和右方多余边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 保存并关闭当前图窗
    save_path = os.path.join(output_dir, f'flatmap_{region_name}.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {save_path}')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'file', 'color']

fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

# 按 file 分组绘制散点图
for file_name, group in df.groupby('file', sort=False):
    # 取出当前分组对应的颜色
    point_color = group['color'].iloc[0]

    ax.scatter(
        group['flat_x'],
        group['flat_y'],
        c=point_color,
        label=file_name,
        s=15,          # 点的大小，可按需调节 (如 10~30)
        alpha=0.8,     # 点的透明度
        edgecolors='none'
    )

# 设置图像属性与反转 Y 轴（如果 flatmap 坐标系原点在左上方，取消下一行注释）
# ax.invert_yaxis()

ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('Soma Distribution Flatmap', fontsize=14)
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 坐标比例

# 添加图例（放在图右侧外部，防止遮挡散点）
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=9,
    markerscale=1.5
)

# 去除多余的顶部和右侧边框 (美化样式)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()